# Ration Early or Run Dry?

## A reservoir trade-off in four TaqSim blocks

Dr. Tobias Siegfried · CC BY 4.0

# The question

A reservoir fills in winter. A farm takes water from it in summer. In most years there is enough. In a dry year
there is not, and the operator has two ways to run the reservoir:

- **Deliver in full until the reservoir is empty.** The farm gets everything it asks for, for as long as possible.
  If the water runs out, the last weeks of summer get almost nothing.
- **Ration early.** When the reservoir is getting low, deliver only part of what the farm asks for. The water lasts
  longer, but the farm is short more often, including in years when the water would have lasted anyway.

Which is better? It depends on what you count. This notebook builds the smallest model that shows it: four TaqSim
blocks, two knobs, two goals. Then it lets the optimizer search.

**What you will take away**

1. How to build and run a small water system in TaqSim, block by block.
2. Why two reasonable goals can pull in opposite directions, and how to see every sensible compromise at once.
3. How to hand the search to the optimizer, and how to read what it returns.

> **The technical words.** Engineers call "deliver in full until empty" the *standard operating policy* and
> "ration early" *hedging*. A plan that no other plan beats on every goal is called *Pareto-optimal*, and the set
> of such plans is the *Pareto front*. This notebook says "unbeaten plans".

# The system in four blocks

TaqSim describes a water system as a **network**: nodes that hold, use or pass on water, joined by edges along
which water flows. Ours has four nodes and three edges.

In [ ]:
# @title Setup: run this cell first
# ── Setup: run this cell first. In Colab it takes about a minute. ──────────────────────────────────
# It fetches the course folder, installs TaqSim, and tells Python where the course code lives.
import os
import sys
import subprocess

if not os.path.exists("src/zarafshan_taqsim"):  # not inside the course folder yet (the normal case in Colab)
    if not os.path.exists("zarafshan-taqsim-course"):
        subprocess.run(["git", "clone", "-q", "--depth", "1", "https://github.com/hydrosolutions/zarafshan-taqsim-course.git"], check=True)
    os.chdir("zarafshan-taqsim-course")
try:
    import taqsim  # noqa: F401  (already installed when you work on your own computer)
except ImportError:
    %pip install -q deap ctrl-freak pymoo
    %pip install -q --no-deps "git+https://github.com/hydrosolutions/taqsim.git@v0.1.4"
sys.path[:0] = [os.path.abspath("src")]
print("Ready. Working folder:", os.getcwd())


In [ ]:
# @title Code: setup
#| label: setup
# Standard tools
import inspect  # to show the source code of a function or class inside this notebook
import logging

import pandas as pd  # tables
from IPython.display import HTML, Markdown  # to show formatted text and HTML as cell output

# TaqSim itself: the four kinds of node we use, the edge that joins them, and the system that holds them
from taqsim import Demand, Edge, Sink, Source, Storage, TimeSeries, WaterSystem
from taqsim.node import NoLoss  # a loss rule that loses nothing (reservoirs need one)
from taqsim.node.events import DeficitRecorded  # the event a Demand node records when it is short
from taqsim.time import Frequency  # the length of one simulation step (monthly here)

# The helpers written for this coursebook (src/zarafshan_taqsim/rationing_coursebook.py)
from zarafshan_taqsim.rationing_coursebook import (
    CAPACITY_MM3,
    EVAPORATION_SHARE_EXAMPLE,
    MONTHLY_NEED_MM3,
    NO_RATIONING,
    PAPER_REGIME,
    START_STORAGE_MM3,
    YEARS,
    Plan,
    RationingRelease,
    SummerEvaporation,
    drought_window_start,
    explorer_html,
    explorer_payload,
    front_of,
    objectives,
    optimise_plans,
    plan_grid,
    plot_front_shift,
    plot_inflows,
    plot_network,
    plot_regime_comparison,
    plot_runs,
    plot_score_maps,
    plot_system_sketch,
    plot_tradeoff,
    score_plans,
    season_statistics,
    simulate_plan,
    synthetic_inflows,
)

logging.basicConfig(level=logging.WARNING)  # only show warnings and errors, not progress messages
pd.set_option("display.precision", 2)  # two decimals in tables

LANG = "en"  # the web page shows all three languages and lets you switch; a notebook is built for one language


def say(en: str, uz: str, ru: str) -> Markdown:
    """Text in three languages: all three on the web page (the switcher picks one), one in a notebook."""
    if LANG:
        return Markdown({"en": en, "uz": uz, "ru": ru}[LANG])
    blocks = (("en", en), ("uz", uz), ("ru", ru))
    return Markdown("\n\n".join(f"::: {{.lang-{code}}}\n{text}\n:::" for code, text in blocks))


def show_source(thing) -> Markdown:
    """Show the source code of a function or class as a code block."""
    return Markdown(f"```python\n{inspect.getsource(thing)}```")

In [ ]:
# @title Code: fig-network
#| label: fig-network
#| fig-cap: "The model as a network. Each node is one TaqSim block; each edge is named after the nodes it joins."
#| echo: false
_ = plot_network()

- **river** is a `Source`: it puts water into the network each month, from a list of numbers.
- **reservoir** is a `Storage`: it keeps water up to its capacity, loses some (if we tell it to), and releases
  some according to a rule. This is where the decisions are made.
- **farm** is a `Demand`: it asks for a given amount each month and records how much it did not get.
- **downstream** is a `Sink`: whatever the farm does not use leaves the network here.

In [ ]:
# @title Code: fig-system
#| label: fig-system
#| fig-cap: "The same four blocks with their numbers. Water moves from left to right."
#| echo: false
_ = plot_system_sketch()

The numbers come from a classic paper on reservoir performance (Hashimoto, Stedinger and Loucks, 1982, p. 17 and
Table 1), with one change. The paper's river peaks in winter. Rivers in Central Asia are fed by snow and glacier
melt: they peak in summer and are lowest in winter. So we swap the paper's two seasons and keep everything else.
[The comparison section](#sec-paper-regime) runs the paper's own river alongside ours. Volumes are in million
cubic metres (Mm³).

| Quantity | Value |
|---|---|
| Reservoir capacity | 40 Mm³ |
| What the farm needs in summer | 45 Mm³ (7.5 Mm³ a month for six months) |
| What the farm needs in winter | 5 Mm³ (0.83 Mm³ a month for six months): a small off-season demand, taken from the paper |
| River inflow in winter | 25 Mm³ on average, standard deviation 10 Mm³ (the paper: 40 and 15) |
| River inflow in summer | 40 Mm³ on average, standard deviation 15 Mm³ (the paper: 25 and 10) |

The summer need (45) is larger than the average summer inflow (40), so the farm depends on water stored in winter.
The reservoir (40) is smaller than the need, and it can only be filled from the low winter flow, so there is
little room for error.

A model year is six winter months followed by six summer months; no calendar dates are attached. Read it as the
hydrological year used in the Zarafshan basin: winter is October to March, summer (the irrigation season) April
to September. Within a season nothing changes from month to month: each season's inflow is spread evenly over
its six months, and so is the farm's need. The only seasonality is winter against summer, as in the paper. The real
monthly shape of the Zarafshan, with its July peak, appears in the sixteen-blocks game and in the simulation
coursebook.

## The river

We do not have a hundred years of measurements, so we make a river: one hundred years of winter and summer inflows
with the averages and spread from the table above. A dry season tends to be followed by another dry season, as in
real rivers.

In [ ]:
# @title Code: river
#| label: river
inflows = synthetic_inflows()  # 100 years x 12 months = 1 200 numbers, each a month's inflow in Mm³

# Check the river against the numbers we asked for: averages, spread and the link from one season to the next
season_statistics(inflows).style.hide(axis="index").format(precision=2)

`synthetic_inflows()` always gives the same river (it uses a fixed random seed), so every result in this notebook
can be reproduced. The table compares the river's statistics with the ones we asked for; they agree to within a
few percent.

In [ ]:
# @title Code: fig-river
#| label: fig-river
#| fig-cap: "One hundred synthetic years. In many years the summer inflow is far below what the farm needs."
#| echo: false
_ = plot_inflows(inflows)

> **Where this differs from the paper.** (1) The seasons are swapped, as explained above. (2) Each season's
> inflow is spread evenly over six monthly steps. (3) The paper derives its operating rules mathematically; we use
> a simple rule with two knobs. (4) The link between seasons is applied to the logarithms of the flows, which is
> an assumption. The numbers in this notebook are therefore our own, not a reproduction of the paper's.

# Build it in TaqSim

## The rule with two knobs

A TaqSim reservoir needs a *release rule*: a small class that says how much water to let out each month. Ours has
two knobs.

- **Trigger** (Mm³): rationing starts when the reservoir holds less than this.
- **Ration** (0 to 1): the share of the summer need that is then delivered.

In [ ]:
# @title Code: rule-source
#| label: rule-source
#| echo: false
show_source(RationingRelease)

Reading the class from the top:

- `@dataclass(frozen=True)` makes it a small, unchangeable record: it holds its two knobs and nothing else.
- `__params__` names the knobs the optimizer may turn, and `__bounds__` says between which values.
- `trigger` and `ration` are the knobs, with their defaults (0 and 1: no rationing).
- `release(...)` is called by TaqSim once per month. It gets the reservoir (`node`, from which it reads the
  current storage), this month's inflow and the time step `t`, and returns the volume to release in Mm³.

A trigger of 0 (or a ration of 1) means no rationing at all.

## Wire the four blocks

Building a TaqSim model always has the same four steps: create an empty system, add the nodes, add the edges,
check and run.

In [ ]:
# @title Code: wire
#| label: wire
# 1. An empty system. One simulation step is one month.
system = WaterSystem(frequency=Frequency.MONTHLY)

# 2. The four nodes. Each has an id (its name in the network) and the settings its kind needs.
system.add_node(
    Source(
        id="river",
        inflow=TimeSeries(values=list(inflows)),  # one value per month, Mm³
    )
)
system.add_node(
    Storage(
        id="reservoir",
        capacity=CAPACITY_MM3,  # 40 Mm³; more than this spills
        initial_storage=START_STORAGE_MM3,  # 20 Mm³ in store when the simulation starts
        release_policy=RationingRelease(trigger=0.0, ration=1.0),  # our rule; these knobs mean no rationing
        loss_rule=NoLoss(),  # no evaporation for now
    )
)
system.add_node(
    Demand(
        id="farm",
        requirement=TimeSeries(values=list(MONTHLY_NEED_MM3) * YEARS),  # the 12 monthly needs, repeated 100 times
    )
)
system.add_node(Sink(id="downstream"))

# 3. The three edges. An edge is named after the two nodes it joins and carries whatever its source lets out.
for source, target in [("river", "reservoir"), ("reservoir", "farm"), ("farm", "downstream")]:
    system.add_edge(Edge(id=f"{source}_to_{target}", source=source, target=target))

# 4. Check that the network is complete (every node connected, every setting valid), then run all 1 200 months.
system.validate()
system.simulate(len(inflows))

Running the system produces no printout. Instead, every node keeps a record of **events**: what it received,
stored, released, lost, or failed to deliver, month by month. The farm records a `DeficitRecorded` event in every
month it gets less than it needs, and each such event carries the missing volume:

In [ ]:
# @title Code: first-read
#| label: first-read
farm = system.nodes["farm"]  # look up a node by its id

# Add up the missing volume over all deficit events, then divide by the number of years
not_delivered = sum(event.deficit for event in farm.events_of_type(DeficitRecorded))
print(f"Water not delivered: {not_delivered / YEARS:.2f} Mm³ per year, on average")

From here on, `simulate_plan(inflows, plan)` does exactly these steps (build, validate, simulate, read the
events) and hands back the results in a tidy form: the storage month by month, the shortage month by month, and
the three scores introduced next.

# First run: deliver in full until empty

In [ ]:
# @title Code: fig-first-run
#| label: fig-first-run
#| fig-cap: "No rationing. Left: water in the reservoir over twenty of the hundred years. Right: for each year, how short the worst month was."
no_rationing = simulate_plan(inflows, NO_RATIONING)  # NO_RATIONING is Plan(trigger=0, ration=1)
first_year = drought_window_start(no_rationing)  # a 20-year window that contains the worst drought

# One row per plan; the right panel shows the 20 years starting in first_year, shaded in the left panel
_ = plot_runs({"Deliver in full until empty": no_rationing}, first_year=first_year)

The left panel shows all 100 years. For each year it draws one bar: the worst month of that year, as the share of
that month's need the farm did not get. A bar of 72 % means that in its worst month the farm received only 28 %
of what it asked for; most years have no bar at all. The right panel zooms into the shaded 20 years around the
worst drought: the reservoir fills every winter (the rising lines) and is drawn down every summer, and in the
driest years it reaches zero.

We score a plan with three numbers:

- **Average shortage**: the water the farm needed but did not get, in Mm³ per year.
- **Worst month**: the share of its need the farm did not get in the single worst month of the hundred years.
- **Years short**: the number of years with at least one short month.

In [ ]:
# @title Code: first-scores
#| label: first-scores
#| echo: false
s = no_rationing.scores
say(
    f"Without rationing the farm is short in **{s.years_short} of 100 years**. On average **{s.mean_shortage:.1f} Mm³ "
    f"per year** is not delivered, which is little. But when it goes wrong, it goes badly wrong: in the worst month "
    f"the farm gets **{100 - s.worst_month:.0f} %** of what it needs ({s.worst_month:.0f} % short).",
    f"Cheklashsiz xoʻjalik **100 yildan {s.years_short} tasida** taqchillik koʻradi. Oʻrtacha **yiliga "
    f"{s.mean_shortage:.1f} mln m³** yetkazib berilmaydi, bu kam. Lekin ishkal chiqqanda, qattiq chiqadi: eng yomon "
    f"oyda xoʻjalik keragining **{100 - s.worst_month:.0f} %** ini oladi ({s.worst_month:.0f} % taqchillik).",
    f"Без лимитирования хозяйство испытывает дефицит в **{s.years_short} из 100 лет**. В среднем не подаётся "
    f"**{s.mean_shortage:.1f} млн м³ в год** — немного. Но когда дело идёт плохо, оно идёт очень плохо: в худшем "
    f"месяце хозяйство получает **{100 - s.worst_month:.0f} %** потребности (дефицит {s.worst_month:.0f} %).",
)

# Second run: ration

Two rationing plans. Both start rationing when the reservoir holds less than 16 Mm³. One cuts deliveries a little
(to 80 %), the other cuts deep (to 50 %).

In [ ]:
# @title Code: fig-second-run
#| label: fig-second-run
#| fig-cap: "Three plans, same river. The grey line marks the trigger."
# A Plan holds the two knobs. Below 16 Mm³ in store, deliver 80 % (lightly) or 50 % (deep) of the summer need.
ration_lightly = simulate_plan(inflows, Plan(trigger=16, ration=0.8))
cut_deep = simulate_plan(inflows, Plan(trigger=16, ration=0.5))

# A dictionary of name -> run; the names become the panel titles
runs = {
    "Deliver in full until empty": no_rationing,
    "Ration lightly (below 16 Mm³, deliver 80 %)": ration_lightly,
    "Cut deep (below 16 Mm³, deliver 50 %)": cut_deep,
}
_ = plot_runs(runs, first_year=first_year)

Look at the bars on the left. "Ration lightly" turns the few tall bars into many short ones of 20 %, but the
tallest bar in the drought stays. Under "Cut deep" no bar is taller than 50 %: the ration itself is the worst that
happens, because the reservoir never runs dry. The price shows in the other two scores.

In [ ]:
# @title Code: tbl-three-plans
#| label: tbl-three-plans
#| tbl-cap: "The three plans, scored."
#| echo: false
pd.DataFrame(
    [
        (name, run.scores.mean_shortage, run.scores.worst_month, run.scores.years_short)
        for name, run in runs.items()
    ],
    columns=["Plan", "Average shortage (Mm³/yr)", "Worst month (% short)", "Years short (of 100)"],
).style.hide(axis="index").format({"Average shortage (Mm³/yr)": "{:.1f}", "Worst month (% short)": "{:.0f}"})

In [ ]:
# @title Code: second-scores
#| label: second-scores
#| echo: false
light, deep = ration_lightly.scores, cut_deep.scores
say(
    f"- **Rationing lightly does not help here.** The worst month is still {light.worst_month:.0f} % short: in the "
    f"worst drought the reservoir runs dry anyway. The farm is simply short more often "
    f"({light.years_short} years instead of {s.years_short}).\n"
    f"- **Cutting deep works, at a price.** The worst month improves from {s.worst_month:.0f} % to "
    f"{deep.worst_month:.0f} % short. But {deep.mean_shortage:.1f} Mm³ per year is not delivered instead of "
    f"{s.mean_shortage:.1f}, and the farm is short in {deep.years_short} years instead of {s.years_short}.\n\n"
    f"So there is no free lunch: a milder worst month costs water. How much, exactly?",
    f"- **Ozgina cheklash bu yerda yordam bermaydi.** Eng yomon oy hali ham {light.worst_month:.0f} % taqchil: eng "
    f"qattiq qurgʻoqchilikda ombor baribir quriydi. Xoʻjalik shunchaki koʻproq taqchillik koʻradi "
    f"({s.years_short} oʻrniga {light.years_short} yil).\n"
    f"- **Keskin kesish ishlaydi, lekin bahosi bor.** Eng yomon oy {s.worst_month:.0f} % dan {deep.worst_month:.0f} % "
    f"taqchillikka yaxshilanadi. Lekin {s.mean_shortage:.1f} oʻrniga yiliga {deep.mean_shortage:.1f} mln m³ "
    f"yetkazilmaydi, va xoʻjalik {s.years_short} oʻrniga {deep.years_short} yilda taqchillik koʻradi.\n\n"
    f"Demak, tekin tushlik yoʻq: yumshoqroq eng yomon oy suvga tushadi. Aniq qanchaga?",
    f"- **Лёгкое лимитирование здесь не помогает.** Худший месяц по-прежнему с дефицитом {light.worst_month:.0f} %: "
    f"в самую сильную засуху водохранилище всё равно пустеет. Хозяйство просто чаще испытывает дефицит "
    f"({light.years_short} лет вместо {s.years_short}).\n"
    f"- **Сильное урезание работает, но имеет цену.** Худший месяц улучшается с {s.worst_month:.0f} % до "
    f"{deep.worst_month:.0f} % дефицита. Но не подаётся {deep.mean_shortage:.1f} млн м³ в год вместо "
    f"{s.mean_shortage:.1f}, и хозяйство испытывает дефицит в {deep.years_short} годах вместо {s.years_short}.\n\n"
    f"Бесплатных обедов нет: более мягкий худший месяц стоит воды. Сколько именно?",
)

## Try your own plan

In [ ]:
# @title Code: fig-my-plan
#| label: fig-my-plan
#| fig-cap: "Your plan against no rationing."
my_plan = Plan(trigger=22, ration=0.55)  # change these two numbers and run the cell again

mine = simulate_plan(inflows, my_plan)
_ = plot_runs({"Deliver in full until empty": no_rationing, "My plan": mine}, first_year=first_year)
print(mine.scores)  # the three scores of your plan

The trigger can be anything from 0 to 40 Mm³ (the capacity), the ration anything from 0 to 1. A trigger of 40
means "always ration in summer".

# Every plan at once

Two knobs are few enough to simply try everything: 21 triggers × 21 rations = 441 plans, 441 TaqSim runs.

In [ ]:
# @title Code: grid
#| label: grid
plans = plan_grid()  # 441 Plans: triggers 0, 2, 4, ... 40 Mm³ combined with rations 0, 0.05, 0.10, ... 1
table = score_plans(inflows, plans)  # runs each plan through TaqSim; one row per plan with its three scores

table.sample(5, random_state=1).sort_index().style.hide(axis="index").format(precision=2)  # five rows, as a taste

Each row is one plan: its two knobs, its three scores, and an `unbeaten` flag explained below. All 441 runs
take a few seconds.

In [ ]:
# @title Code: fig-maps
#| label: fig-maps
#| fig-cap: "Each score for every plan. Darker is worse. The three maps do not agree on where the good plans are."
#| echo: false
_ = plot_score_maps(table)

Now put two of the scores against each other. Every plan becomes one dot.

In [ ]:
# @title Code: fig-tradeoff
#| label: fig-tradeoff
#| fig-cap: "All 441 plans. Left is better (less water undelivered). Down is better (milder worst month). A plan is 'beaten' if another plan is at least as good on both counts and better on one."
# Every row of the table becomes a dot; `named` runs are circled and labelled
_ = plot_tradeoff(table, named={"deliver in full": no_rationing, "ration lightly": ration_lightly, "cut deep": cut_deep})

The black dots are the **unbeaten plans**. For each of them, the only way to get a milder worst month is to leave
more water undelivered. They are the only plans worth discussing; which one to pick is a judgement, not a
calculation.

In [ ]:
# @title Code: tbl-front
#| label: tbl-front
#| tbl-cap: "The unbeaten plans, from least water undelivered to mildest worst month."
#| echo: false
front = front_of(table)
front[["trigger", "ration", "mean_shortage", "worst_month", "years_short"]].rename(
    columns={
        "trigger": "Trigger (Mm³)",
        "ration": "Ration",
        "mean_shortage": "Average shortage (Mm³/yr)",
        "worst_month": "Worst month (% short)",
        "years_short": "Years short (of 100)",
    }
).style.hide(axis="index").format(
    {"Trigger (Mm³)": "{:.0f}", "Ration": "{:.2f}", "Average shortage (Mm³/yr)": "{:.2f}", "Worst month (% short)": "{:.0f}"}
)

# Let the optimizer search

Trying everything worked because there are only two knobs. A real reservoir rule has dozens (different triggers and
rations for each month, several reservoirs, several users). With ten knobs and 21 values each, "everything" is
more runs than anyone can wait for.

The optimizer (TaqSim uses a method called NSGA-II) searches instead. It starts with random plans, keeps the
unbeaten ones, makes new plans by mixing and nudging them, and repeats. We give it the system and the two goals:

In [ ]:
# @title Code: objectives-source
#| label: objectives-source
#| echo: false
show_source(objectives)

Each `Objective` has a name, a direction, and a function that turns a simulated system into one number.

In [ ]:
# @title Code: optimise
#| label: optimise
found = optimise_plans(
    inflows,
    pop_size=40,  # the optimizer works with 40 plans at a time
    generations=40,  # and improves them 40 times: 1 600 TaqSim runs in all
    seed=42,  # the starting plans are random; a fixed seed makes the result reproducible
)
print(f"The optimizer returned {len(found)} plans, {len(front_of(found))} of them different.")

Inside, `optimise_plans` is one call to TaqSim, `optimize(system, objectives, timesteps, pop_size, generations,
seed)`, followed by a re-run of each returned plan to read off its scores. The result table has the same columns
as the grid table above.

In [ ]:
# @title Code: fig-optimizer
#| label: fig-optimizer
#| fig-cap: "The optimizer's plans (blue) on top of the 441 grid plans. It finds the same curve and fills the gaps, because it is not tied to the grid's steps."
_ = plot_tradeoff(table, found=found)

# What to notice

In [ ]:
# @title Code: notice
#| label: notice
#| echo: false
first, last = front.iloc[0], front.iloc[-1]
knee = front.loc[(front["worst_month"] - 50.0).abs().idxmin()]
beaten = int((~table["unbeaten"]).sum())
best_volume = table.loc[table["mean_shortage"].idxmin()]
say(
    f"**1. What you count decides the answer.** If the only goal is to deliver as much water as possible, there is "
    f"one best plan and it is *no rationing* (average shortage {best_volume['mean_shortage']:.2f} Mm³ per year, the "
    f"lowest of all 441 plans). The trade-off only appears once you also care about the worst month. Choosing the "
    f"goals is the most important modelling decision, and it is not a technical one.\n\n"
    f"**2. The curve bends.** Going from {first['worst_month']:.0f} % to {knee['worst_month']:.0f} % short in the "
    f"worst month costs {knee['mean_shortage'] - first['mean_shortage']:.1f} Mm³ per year. Going on from "
    f"{knee['worst_month']:.0f} % to {last['worst_month']:.0f} % costs another "
    f"{last['mean_shortage'] - knee['mean_shortage']:.1f} Mm³ per year. The first steps are cheap, the last ones "
    f"very expensive. A plan near the bend is the natural place to start the discussion.\n\n"
    f"**3. Being short more often is part of the price.** Along the curve the number of short years rises from "
    f"{first['years_short']:.0f} to {last['years_short']:.0f} of 100.\n\n"
    f"**4. Most plans are simply bad.** {beaten} of the 441 plans are beaten, including the reasonable-sounding "
    f"\"ration lightly\". Intuition is a poor guide here; the search is not.",
    f"**1. Nimani hisoblashingiz javobni belgilaydi.** Agar yagona maqsad iloji boricha koʻp suv yetkazish boʻlsa, "
    f"bitta eng yaxshi reja bor va bu — *cheklash yoʻq* (oʻrtacha taqchillik yiliga "
    f"{best_volume['mean_shortage']:.2f} mln m³, barcha 441 reja ichida eng kami). Murosa faqat eng yomon oy ham "
    f"sizga muhim boʻlgandagina paydo boʻladi. Maqsadlarni tanlash — modellashtirishdagi eng muhim qaror, va u "
    f"texnik qaror emas.\n\n"
    f"**2. Egri chiziq bukiladi.** Eng yomon oyda {first['worst_month']:.0f} % dan {knee['worst_month']:.0f} % "
    f"taqchillikka oʻtish yiliga {knee['mean_shortage'] - first['mean_shortage']:.1f} mln m³ ga tushadi. "
    f"{knee['worst_month']:.0f} % dan {last['worst_month']:.0f} % gacha davom etish yana yiliga "
    f"{last['mean_shortage'] - knee['mean_shortage']:.1f} mln m³ ga tushadi. Birinchi qadamlar arzon, oxirgilari "
    f"juda qimmat. Muhokamani bukilish yaqinidagi rejadan boshlash tabiiy.\n\n"
    f"**3. Koʻproq taqchillik koʻrish — bahoning bir qismi.** Egri chiziq boʻylab taqchil yillar soni 100 dan "
    f"{first['years_short']:.0f} tadan {last['years_short']:.0f} tagacha oshadi.\n\n"
    f"**4. Koʻpchilik rejalar shunchaki yomon.** 441 rejadan {beaten} tasi ustun kelingan, shu jumladan oqilona "
    f"tuyulgan \"ozgina cheklash\" ham. Bu yerda sezgi yomon yoʻlboshchi; qidiruv esa yaxshi.",
    f"**1. Ответ определяет то, что вы считаете.** Если единственная цель — подать как можно больше воды, есть один "
    f"лучший план, и это *без лимитирования* (средний дефицит {best_volume['mean_shortage']:.2f} млн м³ в год, "
    f"наименьший из всех 441 планов). Компромисс появляется только тогда, когда вам важен и худший месяц. Выбор "
    f"целей — самое важное решение при моделировании, и оно не техническое.\n\n"
    f"**2. Кривая изгибается.** Переход от {first['worst_month']:.0f} % к {knee['worst_month']:.0f} % дефицита в "
    f"худшем месяце стоит {knee['mean_shortage'] - first['mean_shortage']:.1f} млн м³ в год. Дальнейший переход от "
    f"{knee['worst_month']:.0f} % к {last['worst_month']:.0f} % стоит ещё "
    f"{last['mean_shortage'] - knee['mean_shortage']:.1f} млн м³ в год. Первые шаги дёшевы, последние очень дороги. "
    f"План возле изгиба — естественная отправная точка для обсуждения.\n\n"
    f"**3. Более частый дефицит — часть цены.** Вдоль кривой число дефицитных лет растёт с "
    f"{first['years_short']:.0f} до {last['years_short']:.0f} из 100.\n\n"
    f"**4. Большинство планов просто плохи.** {beaten} из 441 планов превзойдены, включая разумное на слух «лёгкое "
    f"лимитирование». Интуиция здесь плохой советчик; поиск — хороший.",
)

# One more block: evaporation

So far, water saved in the reservoir waits there for free. In a hot climate it does not: some of it evaporates. In
TaqSim that is one more small class, a *loss rule*, plugged into the reservoir:

In [ ]:
# @title Code: evaporation-source
#| label: evaporation-source
#| echo: false
show_source(SummerEvaporation)

`calculate(...)` is called by TaqSim once per month, after the inflow has been stored and before the release. It
returns what was lost and why; TaqSim subtracts it from the storage and records a `WaterLost` event.

In [ ]:
# @title Code: fig-evaporation
#| label: fig-evaporation
#| fig-cap: "The unbeaten plans without and with evaporation. With evaporation every plan delivers less, and a mild worst month becomes much more expensive."
# Same 441 plans, same river, but now the reservoir loses 3 % of its content in each summer month
with_evaporation = score_plans(inflows, plan_grid(), evaporation_share=EVAPORATION_SHARE_EXAMPLE)

# Draw the unbeaten plans of both tables on one chart
_ = plot_front_shift({"no evaporation": table, "3 % of the stored water evaporates each summer month": with_evaporation})

In [ ]:
# @title Code: evaporation-text
#| label: evaporation-text
#| echo: false
front_e = front_of(with_evaporation)
at_50 = front[front["worst_month"] <= 50.0].iloc[0]
at_50_e = front_e[front_e["worst_month"] <= 50.0].iloc[0]
say(
    f"The 3 % is an illustrative figure, not a measurement. The effect is large: holding the worst month to 50 % "
    f"short costs {at_50['mean_shortage']:.1f} Mm³ per year without evaporation and "
    f"{at_50_e['mean_shortage']:.1f} Mm³ per year with it. Saving water for later is worth less when the saved "
    f"water does not keep.",
    f"3 % — oʻlchov emas, tushuntirish uchun olingan raqam. Taʼsiri katta: eng yomon oyni 50 % taqchillikda "
    f"ushlab turish bugʻlanishsiz yiliga {at_50['mean_shortage']:.1f} mln m³, bugʻlanish bilan esa yiliga "
    f"{at_50_e['mean_shortage']:.1f} mln m³ ga tushadi. Saqlangan suv saqlanmasa, uni keyinga asrash kamroq "
    f"qadrli.",
    f"3 % — иллюстративное число, не измерение. Эффект велик: удержать худший месяц на уровне 50 % дефицита стоит "
    f"{at_50['mean_shortage']:.1f} млн м³ в год без испарения и {at_50_e['mean_shortage']:.1f} млн м³ в год с "
    f"ним. Беречь воду на потом менее выгодно, когда сбережённая вода не сохраняется.",
)

# Another river: the paper's regime

The paper's river peaks in winter and is lowest in summer, the opposite of ours. Same reservoir, same farm, same
hundred random draws; only the seasons are swapped. Does the trade-off look the same?

In [ ]:
# @title Code: fig-regimes
#| label: fig-regimes
#| fig-cap: "Left: the average month of both rivers against the farm's need. Right: the unbeaten plans of both. Same reservoir, same need, same random seed."
paper_inflows = synthetic_inflows(regime=PAPER_REGIME)  # the paper's seasons: winter 40 Mm³, summer 25 Mm³
paper_table = score_plans(paper_inflows, plan_grid())  # the same 441 plans on that river

_ = plot_regime_comparison(
    {"Central Asian regime (this notebook)": (inflows, table), "Paper regime (winter peak)": (paper_inflows, paper_table)}
)

In [ ]:
# @title Code: regimes-text
#| label: regimes-text
#| echo: false
paper_front = front_of(paper_table)
paper_base = simulate_plan(paper_inflows, NO_RATIONING).scores
ours_50 = front[front["worst_month"] <= 50.0].iloc[0]
paper_50 = paper_front[paper_front["worst_month"] <= 50.0].iloc[0]
say(
    f"- **Same shape.** Both curves bend the same way: cheap at first, expensive at the end. The example works for "
    f"either river.\n"
    f"- **The paper's river is the harder case.** Without rationing it is {paper_base.worst_month:.0f} % short in "
    f"its worst month (ours: {s.worst_month:.0f} %), and holding the worst month to 50 % costs "
    f"{paper_50['mean_shortage'] - paper_base.mean_shortage:.1f} Mm³ per year there against "
    f"{ours_50['mean_shortage'] - s.mean_shortage:.1f} here. With a summer peak the water arrives when the farm "
    f"needs it, and the reservoir only has to cover the gap.\n"
    f"- **What runs our reservoir dry** is a poor winter (little to store) followed by a weak melt. That is the "
    f"Central Asian pattern: the reservoir is filled from the low winter flow, as Kattakurgan is.",
    f"- **Shakli bir xil.** Ikkala egri chiziq ham bir xil bukiladi: avval arzon, oxirida qimmat. Misol ikkala daryo "
    f"uchun ham ishlaydi.\n"
    f"- **Maqoladagi daryo — qiyinroq holat.** Cheklashsiz uning eng yomon oyi {paper_base.worst_month:.0f} % "
    f"taqchil (bizniki: {s.worst_month:.0f} %), va eng yomon oyni 50 % da ushlab turish u yerda yiliga "
    f"{paper_50['mean_shortage'] - paper_base.mean_shortage:.1f} mln m³, bizda esa "
    f"{ours_50['mean_shortage'] - s.mean_shortage:.1f} mln m³ ga tushadi. Yozgi toshqin bilan suv xoʻjalikka kerak "
    f"paytda keladi, va ombor faqat farqni qoplashi kerak.\n"
    f"- **Bizning omborni quritadigan narsa** — kam suvli qish (saqlashga kam suv) va undan keyingi kuchsiz erish. "
    f"Bu Markaziy Osiyo manzarasi: ombor, Kattaqoʻrgʻon suv ombori kabi, kam suvli qishki oqimdan toʻldiriladi.",
    f"- **Та же форма.** Обе кривые изгибаются одинаково: сначала дёшево, в конце дорого. Пример работает для обеих "
    f"рек.\n"
    f"- **Река из статьи — более трудный случай.** Без лимитирования её худший месяц имеет дефицит "
    f"{paper_base.worst_month:.0f} % (у нашей: {s.worst_month:.0f} %), а удержание худшего месяца на уровне 50 % "
    f"стоит там {paper_50['mean_shortage'] - paper_base.mean_shortage:.1f} млн м³ в год против "
    f"{ours_50['mean_shortage'] - s.mean_shortage:.1f} у нас. При летнем пике вода приходит тогда, когда она нужна "
    f"хозяйству, и водохранилищу остаётся лишь покрыть разницу.\n"
    f"- **Наше водохранилище опустошает** скудная зима (мало что накопить), за которой следует слабое таяние. Это "
    f"центральноазиатская картина: водохранилище наполняется из скудного зимнего стока, как Каттакурганское.",
)

# From this toy to a real reservoir

- **Real rules have the same knobs, many times over.** The release rule of the Kattakurgan reservoir in the
  Zarafshan model of this repository (`ZRBReleaseRule`) has a storage level below which releases are cut (`v1`) and
  a cut factor (`buffer_coef`): our trigger and ration, with a separate value for each month. That is why the
  full model needs the optimizer.
- **What this toy leaves out.** Forecasts of the coming season; several users with different priorities; how crops
  actually respond to a shortage; evaporation that depends on the lake's surface area; return flows.
- **One river is one river.** The numbers here belong to this one synthetic series of a hundred years. Other
  series give other numbers. The shape (a bent curve from "no rationing" to "cut deep") was the same for the three
  other series we tested.

# Exercises

1. **A bigger reservoir.** In `rationing_coursebook.py`, set `CAPACITY_MM3 = 60`. Does rationing still pay? Where
   is the bend now?
2. **A different second goal.** Replace "worst month" with "years short" in `objectives()` and run the optimizer
   again. Which plans are unbeaten now? Is "no rationing" still one of them?
3. **A different river.** Use `synthetic_inflows(seed=1)`, then `synthetic_inflows(regime=PAPER_REGIME, seed=1)`.
   Which numbers change, and what stays the same?
4. **A seasonal rule.** Give the rule two rations, one for early and one for late summer. Does the optimizer find
   plans that beat the two-knob curve?

# Sources

- Hashimoto, T., Stedinger, J. R., Loucks, D. P. (1982). Reliability, resiliency, and vulnerability criteria for
  water resource system performance evaluation. *Water Resources Research* 18(1), 14–20. The reservoir, its
  numbers (p. 17, Table 1; seasons swapped here) and the trade-off between failing often and failing badly
  (Fig. 9).
- Draper, A. J., Lund, J. R. (2004). Optimal hedging and carryover storage value. *Journal of Water Resources
  Planning and Management* 130(1), 83–87. When rationing pays (p. 83) and why a rule with two parameters is often
  enough (p. 87).
- Deb, K., Pratap, A., Agarwal, S., Meyarivan, T. (2002). A fast and elitist multiobjective genetic algorithm:
  NSGA-II. *IEEE Transactions on Evolutionary Computation* 6(2), 182–197. The search method behind
  `taqsim.optimize`.